# Terms
Before training, let's explain every term that is commonly used in machine learning.

#### Batches
Batches means how many batches we sepearate the training data into, based on `batch_size`. For example, if the training data has size of 128 and `batch_size` is 32, then it means we will have `128 / 32 = 4` batches. During training, we will feed each batch (size 32) into the model to perform forward pass, then evalaute the losses and update the weights based on some optimizers (such as SGD (stochastic gradient descent)). We will continue until all batches have been fed into the model, and we counted it as 1 epoch.

#### Padding
Padding is a process of adding special tokens into the data so that every data has the same size. It is necessary beacuse our model needs to have fixed size of input, and fixed size of output. For example, if the vocab is `{"hello": 1, "world": 2, "!": 3, "<pad>": 0}`, and there are two data:
- `hello hello world!`
- `hello!`

We would like to make sure the data has the same size (for example, size 4). Then we will transform the data into:
- `["hello", "hello", "world", "!"]`
- `["hello", "!", "<pad>", "<pad>"]`

Notice that we added a special tokens `<pad>` such that both data have the same size.

# Parameters
Some parameters in our transformer model (in real life example there will be more parameters):
- `d_model`: The dimension or length of the embedding vectors
- `heads`: The number of heads of attention
- `vocab_size_source`: The number of tokens in the source vocabulary
- `vocab_size_target`: The number of tokens in the target vocabulary
- `num_encoder_layers`: How many layers of encoders that we want the model to have
- `num_decoder_layers`: How many layers of decoders that we want the model to have
- `max_seq_len`: The maximum size of the input sequence length
- `d_ff`: The number of neurons in the hidden layer of the encounders and decoders

# Review
Let's do a final review on each part of the transformer, and make sure we have complete understanding of the shapes on each step:

### Encoder
- Input Embedding: Input embeddings are batches of data with `seq_len` that are converted into embeddings. For example: `[[[0.5, 1, 2], [1, 2, 3]], [[0.8, 0.9, 0.11], [1.2, 1.4, -1.5]]]`, in this case, we have 2 batches of input, with `seq_len=2` and their corresponding embeddings are: `[[0.5, 1, 2], [1, 2, 3]]` and `[[0.8, 0.9, 0.11], [1.2, 1.4, -1.5]]`.
    - Input size: `(batch_size, seq_len)`
    - Output size: `(batch_size, seq_len, d_model)`
- Positional Encoding: Add positional encoding to the input embedding so that the model can understand what are the positions of different tokens.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Multi-Head Attention: Compute Q, K, and V base on the input embeddings. Even though we divided them into `h` heads attention, in the end when we concat them together, the size should be unchanged.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Add & Norm: Add the output of the multi-head attention with the original input embedding, and normalize it. This also shouldn't change the size.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Feed Forward: Feed the output of the previous add&norm result to a neural network with hidden layers.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Second add&norm: Perform add&norm again.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`

### Decoder
- Input Embedding: Input embeddings are batches of data with `seq_len` that are converted into embeddings. For example: `[[[0.5, 1, 2], [1, 2, 3]], [[0.8, 0.9, 0.11], [1.2, 1.4, -1.5]]]`, in this case, we have 2 batches of input, with `seq_len=2` and their corresponding embeddings are: `[[0.5, 1, 2], [1, 2, 3]]` and `[[0.8, 0.9, 0.11], [1.2, 1.4, -1.5]]`.
    - Input size: `(batch_size, seq_len)`
    - Output size: `(batch_size, seq_len, d_model)`
- Positional Encoding: Add positional encoding to the input embedding so that the model can understand what are the positions of different tokens.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Masked Multi-head Attention: Just like multi-head attention, compute Q, K, and V, but after that we apply something called `mask` so that the model can only get information from tokens that are prior to the current token. This prevents the model from "seeing" the future.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Add & Norm: Add the output of the masked multi-head attention with the original input embedding, and normalize it. This also shouldn't change the size.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Multi-Head Attention: We use the decoder attention's Q, the encoder attention's K, and V here. The reason for this specific configuration of Query, Key, and Value sources is precisely for the decoder to gain context about the overall source sentence based on the rich representation produced by the encoder.
    - Input size from encoder: `(batch_size, encoder_seq_len, d_model)`
    - Input size from decoder: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Second add&norm: Perform add&norm again.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Feed Forward: Feed the output of the previous add&norm result to a neural network with hidden layers.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Third add&norm: Perform add&norm again.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, d_model)`
- Output layer (linear): The output layer return logits to represent the probabilities for each token in the vocabulary at each position in the target sequence.
    - Input size: `(batch_size, seq_len, d_model)`
    - Output size: `(batch_size, seq_len, vocab_size)`

# Translation Model
To have a better understanding of the transformer model, let's train a small model to translate English into Chinese. This model will consists of both the encoder and the decoder. Let's first copy the code we wrote previously for encoder and decoder.

In [1]:
import torch
from torch import nn

class EncoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        d_ff: int = None,
        dropout: float = 0.1
    ):
        super().__init__()
        # The same as the attention that we talked about before
        # but pytorch has it ready and we don't need to implement
        # on our own
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm((d_model, ))
        # A convention is to have 4 * d_model as the output shape
        d_ff = d_model * 4 if not d_ff else d_ff
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm((d_model, ))
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, X, src_padding_mask):
        """X has shape (batch_size, sequence_length, d_model)"""
        # Apply multi-head attention to get new representation
        attn_output, _ = self.attention(X, X, X, key_padding_mask=src_padding_mask)
        # Add & Norm for the multi-head attention output
        norm1_input = X + attn_output
        norm1_output = self.norm1(norm1_input)
        # Feed forward
        ffn_output = self.linear_relu_stack(norm1_output)
        ffn_output = self.dropout2(ffn_output)
        # Add & Norm for the FFN output
        norm2_input = norm1_output + ffn_output
        norm2_output = self.norm2(norm2_input)
        return norm2_output

In [2]:
class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        d_ff: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        # The masked Multi-head attention
        self.masked_self_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm((d_model, ))
        self.dropout1 = nn.Dropout(dropout)
        # The attention that came from the encoder
        self.encoder_decoder_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm((d_model, ))
        self.dropout2 = nn.Dropout(dropout)
        # The feed forward (FFN) part
        # A convention is to have 4 * d_model as the output shape
        d_ff = d_model * 4 if not d_ff else d_ff
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm3 = nn.LayerNorm(d_model) # Normalizes over the d_model dimension
        self.dropout3 = nn.Dropout(dropout) # Dropout after FFN residual

    def forward(
        self,
        target_input,
        encoder_output,
        src_padding_mask=None,
        trg_padding_mask=None,
        trg_attn_mask=None,
    ):
        """
        Decoder will need to take in the encoder output as part of the input
        """
        residual_input = target_input
        # --- Masked Multi-Head Self-Attention ---
        # Query, Key, Value are the same (from the decoder's path)
        # Pass the trg_attn_mask (causal), and trg_padding_mask
        masked_attn_output, _ = self.masked_self_attention(
            query=target_input,
            key=target_input,
            value=target_input,
            attn_mask=trg_attn_mask,
            key_padding_mask=trg_padding_mask,
        )

        # --- Add & Norm 1 ---
        # Add residual connection (input to this sub-layer)
        output_after_self_attn = residual_input + masked_attn_output
        # Apply dropout
        output_after_self_attn = self.dropout1(output_after_self_attn)
        # Apply Layer Norm
        norm1_output = self.norm1(output_after_self_attn) # Shape (batch_size, target_seq_len, d_model)

        # Store input for next residual connection
        residual_input = norm1_output # Input to encoder-decoder attention

        # --- Multi-Head Encoder-Decoder Attention (Cross-Attention) ---
        # Query from decoder's path (output of first norm)
        # Key and Value from encoder's output
        # src_padding_mask can be used here to mask padding in the encoder output
        cross_attn_output, _ = self.encoder_decoder_attention(
            query=norm1_output,
            key=encoder_output,
            value=encoder_output,
            key_padding_mask=src_padding_mask # Apply encoder padding mask here if needed
        )

        # --- Add & Norm 2 ---
        # Add residual connection (input to this sub-layer)
        output_after_cross_attn = residual_input + cross_attn_output
        # Apply dropout
        output_after_cross_attn = self.dropout2(output_after_cross_attn)
        # Apply Layer Norm
        norm2_output = self.norm2(output_after_cross_attn) # Shape (batch_size, target_seq_len, d_model)

        # Store input for next residual connection
        residual_input = norm2_output # Input to Feed-Forward Network

        # --- Feed-Forward Network ---
        # FFN operates independently on the last dimension
        ffn_output = self.linear_relu_stack(norm2_output) # Shape (batch_size, target_seq_len, d_model)
        # Apply dropout
        ffn_output = self.dropout3(ffn_output) # Dropout after FFN output

        # --- Add & Norm 3 ---
        # Add residual connection (input to this sub-layer)
        output_after_ffn = residual_input + ffn_output
        # Apply Layer Norm
        norm3_output = self.norm3(output_after_ffn) # Shape (batch_size, target_seq_len, d_model)

        return norm3_output # Output of the Decoder Block

In [3]:
import math

class EncoderDecoderTransformer(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        vocab_size_source: int,
        vocab_size_target: int,
        num_encoder_layers: int,
        num_decoder_layers: int,
        max_seq_len: int,
        d_ff: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.dropout = dropout
        self.d_model = d_model
        # Embeddings
        self.source_embeddings = nn.Embedding(vocab_size_source, d_model)
        self.target_embeddings = nn.Embedding(vocab_size_target, d_model)
        # Positional encoding is fixed, therefore we register it in buffer
        pe = self._generate_fixed_positional_encoding(max_seq_len, d_model)
        self.register_buffer('positional_encoding', pe)
        # Encoder layers
        self.encoder_stack = nn.ModuleList([
            EncoderBlock(d_model, heads, d_ff, dropout) for _ in range(num_encoder_layers)
        ])
        # Decoder layers
        self.decoder_stack = nn.ModuleList([
            DecoderBlock(d_model, heads, d_ff, dropout) for _ in range(num_decoder_layers)
        ])
        # Output layer
        self.output_layer = nn.Linear(d_model, vocab_size_target)
        # Initialize weights
        self._initialize_parameters()

    def _initialize_parameters(self):
        # Common initialization for Transformer weights
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _generate_fixed_positional_encoding(self, max_seq_len: int, d_model: int):
        """Generate a matrix for fixed positional encoding base on the max_seq_len"""
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Add batch dimension (1, max_seq_len, d_model)
        return pe # This will be added to embeddings

    def forward(
        self,
        src_tokens,
        trg_tokens,
        trg_padding_mask=None,
        src_padding_mask=None
    ):
        """
        This is a high-level forward pass outline. Actual implementation needs mask handling.
        Masks need to be generated based on src_tokens and trg_tokens padding
        Lookahead mask for decoder self-attention also needs to be generated.
        """
        # 1. Get embeddings and add positional encoding
        src_embed = self.source_embeddings(src_tokens) * math.sqrt(self.d_model) # Scale embeddings
        trg_embed = self.target_embeddings(trg_tokens) * math.sqrt(self.d_model) # Scale embeddings

        # Add positional encoding (need to handle sequence length correctly)
        src_embed = src_embed + self.positional_encoding[:, :src_tokens.size(1), :].to(src_embed.device)
        trg_embed = trg_embed + self.positional_encoding[:, :trg_tokens.size(1), :].to(trg_embed.device)

        # Apply dropout (common)
        src_embed = nn.Dropout(self.dropout)(src_embed)
        trg_embed = nn.Dropout(self.dropout)(trg_embed)

        # 2. Encoder Pass
        encoder_output = src_embed
        for encoder_layer in self.encoder_stack:
            # Pass padding mask to encoder self-attention
            encoder_output = encoder_layer(encoder_output, src_padding_mask=src_padding_mask)

        # 3. Decoder Pass
        decoder_output = trg_embed
        # Generate causal mask for decoder self-attention (shape seq_len, seq_len)
        causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(trg_tokens.size(1)).to(trg_tokens.device)
        # Combine causal mask with target padding mask if needed

        for decoder_layer in self.decoder_stack:
            # Pass target causal mask to masked self-attention
            # Pass encoder padding mask to encoder-decoder attention
            decoder_output = decoder_layer(
                decoder_output,
                encoder_output,
                trg_attn_mask=causal_mask,
                trg_padding_mask=trg_padding_mask,
                src_padding_mask=src_padding_mask # Pass the encoder padding mask
            )

        # 4. Final Output Layer
        # Project d_model to vocab_size
        output_logits = self.output_layer(decoder_output)

        return output_logits # Shape (batch_size, target_sequence_length, vocab_size_target)

Next, let's write a custom dataset that loads in the English and Chinese text line by line.

In [4]:
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, src_path: str, tgt_path: str):
        with open(src_path, 'r', encoding='utf-8') as f:
            self.src_lines = [line.strip() for line in f]
        with open(tgt_path, 'r', encoding='utf-8') as f:
            self.tgt_lines = [line.strip() for line in f]
        assert len(self.src_lines) == len(self.tgt_lines), "Mismatch in number of lines"

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        src = self.src_lines[idx]
        tgt = self.tgt_lines[idx]
        return (src, tgt)

In [5]:
dataset = TranslationDataset(
    src_path="./data/en-zh.txt/TED2013.en-zh.en",
    tgt_path="./data/en-zh.txt/TED2013.en-zh.zh",
)
dataset[0:5]

(['http://www.ted.com/talks/stephen_palumbi_following_the_mercury_trail.html',
  "There's a tight and surprising link between the ocean's health and ours, says marine biologist Stephen Palumbi. He shows how toxins at the bottom of the ocean food chain find their way into our bodies, with a shocking story of toxic contamination from a Japanese fish market. His work points a way forward for saving the oceans' health -- and humanity's.",
  'fish,health,mission blue,oceans,science',
  '899',
  'Stephen Palumbi: Following the mercury trail'],
 ['http://www.ted.com/talks/lang/zh-cn/stephen_palumbi_following_the_mercury_trail.html',
  '生物学家史蒂芬·帕伦认为，海洋的健康和我们的健康之间有着紧密而神奇的联系。他通过日本一个渔场发生的让人震惊的有毒污染的事件，展示了位于海洋食物链底部的有毒物质是如何进入我们的身体的。他的工作主要是未来拯救海洋健康的方法——同时也包括人类的。',
  'fish,health,mission blue,oceans,science',
  '899',
  '史蒂芬·帕伦：追寻水银的踪迹'])

Then we would like to:
1. Split the data into training and test set (in real life we want validation set as well, but for simplicity we are ignoring it here)
2. Wrap the dataset with `DataLoader`, perform shuffling and batching

In [6]:
from torch.utils.data import DataLoader, random_split

train_data, test_data = random_split(dataset, [0.8, 0.2])
print("Training data size:", len(train_data), ", Test data size:", len(test_data))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=True)

sample_train_input, sample_train_output = next(iter(train_loader))
print(sample_train_input[0])
print(sample_train_output[0])

Training data size: 123664 , Test data size: 30915
Because it's so hard to admit our own fallibility.
因为我们很难 我们自己是很容易犯错的


Next let's load in the tokenizer that we trained before, notice that we use `PreTrainedTokenizerFast` instead of `Tokenizer` from the `tokenizers` library because it has more features that we need such as padding, truncation, and so on.

In [7]:
from transformers import PreTrainedTokenizerFast

en_tokenizer = PreTrainedTokenizerFast(tokenizer_file="./output/ted-english.json")
en_tokenizer.add_special_tokens({'pad_token': '[PAD]'})

zh_tokenizer = PreTrainedTokenizerFast(tokenizer_file="./output/ted-chinese.json")
zh_tokenizer.add_special_tokens({'pad_token': '[PAD]'})
# BOS means begining of the sentence, it is needed for the "shifted right" in the decoder
zh_tokenizer.add_special_tokens({'bos_token' : '[BOS]'})

# Padding base on max_length, it will keep adding <PAD> until it reaches max_length
print(en_tokenizer("You smell money", return_tensors="pt", padding="max_length", max_length=16))
print(zh_tokenizer("你闻到的是钱的气味", return_tensors="pt", padding="max_length", max_length=16))

/Users/dxu/Desktop/personal/transformer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'input_ids': tensor([[ 358, 4013,  934,    3,    3,    3,    3,    3,    3,    3,    3,    3,
            3,    3,    3,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}
{'input_ids': tensor([[  337, 20819,  4777, 19520, 11323,     3,     3,     3,     3,     3,
             3,     3,     3,     3,     3,     3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}


In [8]:
import torch

def train_one_epoch(transformer, train_loader, en_tokenizer, zh_tokenizer, optimizer, criterion, device, max_seq_len):
    running_loss = 0.0
    total_batches = len(train_loader)

    pad_token_id = zh_tokenizer.pad_token_id
    sos_token_id = zh_tokenizer.bos_token_id

    if pad_token_id is None or sos_token_id is None:
         raise ValueError("Target tokenizer must have pad_token_id and bos_token_id defined.")

    transformer.train() # Set model to training mode
    for input_seq, output_seq in train_loader:
        optimizer.zero_grad()

        # Tokenization
        # input_seq shape (batch_size, max_seq_len)
        # output_seq shape (batch_size, max_seq_len)
        en_tokens = en_tokenizer(
            input_seq, return_tensors="pt", padding="max_length", truncation=True, max_length=max_seq_len
        ).to(device)
        zh_tokens = zh_tokenizer(
            output_seq, return_tensors="pt", padding="max_length", truncation=True, max_length=max_seq_len
        ).to(device)
        en_input_ids = en_tokens["input_ids"]
        zh_input_ids = zh_tokens["input_ids"]

        # Prepare padding mask
        encoder_padding_mask = torch.where(
            en_tokens["attention_mask"] == 0,
            torch.tensor(float('-inf'), device=device),
            torch.tensor(0.0, device=device),
        )
        decoder_padding_mask = torch.where(
            zh_tokens["attention_mask"] == 0,
            torch.tensor(float('-inf'), device=device),
            torch.tensor(0.0, device=device),
        )

        # Prepare Decoder Input (shifted right: <SOS> + target tokens[:-1])
        batch_size = zh_input_ids.size(0)
        decoder_input_ids = torch.full(
            (batch_size, 1), sos_token_id, device=device, dtype=zh_input_ids.dtype
        )
        decoder_input_ids = torch.cat([decoder_input_ids, zh_input_ids[:, :-1]], dim=1)

        # Target for loss is the original Chinese input IDs
        target_tokens_for_loss = zh_input_ids

        # Forward pass
        # The main model's forward method is assumed to accept these additive masks
        # and handle causal mask generation internally, combining it with decoder_padding_mask for decoder self-attention
        # and passing encoder_padding_mask to decoder cross-attention.
        predictions = transformer(
            src_tokens=en_input_ids,
            trg_tokens=decoder_input_ids, # Pass the shifted decoder input
            src_padding_mask=encoder_padding_mask, # Additive mask for encoder self-attention
            trg_padding_mask=decoder_padding_mask, # Additive mask for decoder self-attention (combined with causal) and cross-attention
        )

        # Calculate Loss
        # CrossEntropyLoss expects input (N, C) and target (N)
        # It will ignore the pad_token_id in target_tokens_for_loss
        loss = criterion(
            predictions.view(-1, predictions.size(-1)),
            target_tokens_for_loss.view(-1)
        )

        # Backpropagation and Optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / total_batches

In [ ]:
import torch
from torch import nn
import torch.optim as optim

num_epochs = 1
max_seq_len = 8
device = torch.device("cpu")
transformer_model = EncoderDecoderTransformer(
    d_model=64,
    heads=2,
    vocab_size_source=len(en_tokenizer),
    vocab_size_target=len(zh_tokenizer),
    num_encoder_layers=1,
    num_decoder_layers=1,
    max_seq_len=max_seq_len,
    d_ff=128,
    dropout=0.1
)
optimizer = optim.Adam(transformer_model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=zh_tokenizer.pad_token_id) # Ignore padding in loss

transformer_model.to(device) # Move model to device

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Train for one epoch
    # transformer, train_loader, en_tokenizer, zh_tokenizer, optimizer, criterion, device, max_seq_len
    train_loss = train_one_epoch(
        transformer_model,
        train_loader,
        en_tokenizer,
        zh_tokenizer,
        optimizer,
        criterion,
        device,
        max_seq_len,
    )
    print(f"Train Loss: {train_loss:.4f}")

print("Training finished.")

Because it is too slow to train the model at local given that I don't have any GPUs, we will instead train it on [Google Colab](https://colab.research.google.com/)